# Train FaceNet + SVM on the secondary dataset

This notebook trains and evaluates a face classifier using `dataset/secondary_processed`. It keeps every generated artifact separate from the main attendance model.

> The class IDs in this dataset are anonymized experimental labels, not real student roll numbers.

## 0. Install dependencies in the active notebook kernel

Run the following cell once before the imports. `%pip` installs packages into the Python environment used by this notebook, avoiding a mismatch between the terminal and Jupyter kernel. Python 3.10 is recommended because that is the version specified by this project. Restart the kernel after installation if Jupyter requests it.

In [3]:
from pathlib import Path
import sys

setup_root = Path.cwd().resolve()
if not (setup_root / "requirements.txt").exists() and (setup_root.parent / "requirements.txt").exists():
    setup_root = setup_root.parent

requirements_file = setup_root / "requirements.txt"
if not requirements_file.exists():
    raise FileNotFoundError(f"Could not find requirements.txt from {Path.cwd()}")

print(f"Kernel Python: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
if sys.version_info[:2] != (3, 10):
    print("Warning: this project recommends a Python 3.10 Jupyter kernel.")

%pip install -r "$requirements_file"

Kernel Python: /home/mehedinaeem/Desktop/Code/Bitol_Computer_Vision_System/.venv/bin/python
Python version: 3.14.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 3.8 MB/s  0:00:04m 3.8 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 1.4 MB/s  0:00:031.4 MB/s eta 0:00:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 2.5 MB/s  0:00:022.5 MB/s eta 0:00:01:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following versions that require a dif

## 1. Imports and configuration

Run the notebook from the repository root or from the `notebooks` directory. The first FaceNet call may download model weights if they are not already cached.

In [1]:
from pathlib import Path
import json
import random
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from deepface import DeepFace
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import SVC

SEED = 42
TEST_SIZE = 0.20
MODEL_NAME = "Facenet"
DETECTOR_BACKEND = "skip"  # images are already prepared face crops
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
FORCE_REEXTRACT = False

random.seed(SEED)
np.random.seed(SEED)

candidate_root = Path.cwd().resolve()
if not (candidate_root / "dataset").exists() and (candidate_root.parent / "dataset").exists():
    candidate_root = candidate_root.parent
PROJECT_ROOT = candidate_root

DATASET_DIR = PROJECT_ROOT / "dataset" / "secondary_processed"
MODEL_DIR = PROJECT_ROOT / "models" / "secondary"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports" / "secondary"
EMBEDDINGS_PATH = MODEL_DIR / "embeddings.pkl"
CLASSIFIER_PATH = MODEL_DIR / "face_classifier.pkl"
ENCODER_PATH = MODEL_DIR / "label_encoder.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset:      {DATASET_DIR}")
print(f"Models:       {MODEL_DIR}")
print(f"Reports:      {REPORT_DIR}")

ModuleNotFoundError: No module named 'joblib'

## 2. Index and validate the dataset

In [ ]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_DIR}")

records = []
for class_dir in sorted(path for path in DATASET_DIR.iterdir() if path.is_dir()):
    for image_path in sorted(class_dir.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in VALID_EXTENSIONS:
            records.append({"image_path": str(image_path), "label": class_dir.name})

image_df = pd.DataFrame(records)
if image_df.empty:
    raise ValueError(f"No supported images found under {DATASET_DIR}")

class_counts = image_df["label"].value_counts().sort_index()
if len(class_counts) < 2:
    raise ValueError("SVM training requires at least two classes.")
if class_counts.min() < 2:
    raise ValueError("Every class needs at least two images for a stratified split.")

summary_path = DATASET_DIR / "dataset_summary.csv"
dataset_summary = pd.read_csv(summary_path, dtype={"class_id": str}) if summary_path.exists() else None

print(f"Classes: {len(class_counts)}")
print(f"Images:  {len(image_df)}")
print(f"Images per class: min={class_counts.min()}, max={class_counts.max()}, mean={class_counts.mean():.1f}")
display(image_df.head())
if dataset_summary is not None:
    display(dataset_summary.head())

## 3. Quick visual inspection

In [ ]:
sample_rows = image_df.groupby("label", group_keys=False).sample(n=1, random_state=SEED).head(12)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for axis, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    with Image.open(row["image_path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(row["label"])
    axis.axis("off")
for axis in axes.flat[len(sample_rows):]:
    axis.axis("off")
plt.suptitle("One sample from each of the first 12 classes")
plt.tight_layout()
plt.show()

ax = class_counts.plot(kind="bar", figsize=(14, 4), title="Images per class")
ax.set_xlabel("Class ID")
ax.set_ylabel("Images")
ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.show()

## 4. Extract or load FaceNet embeddings

Embedding extraction is the slowest step. Successful results are cached in `models/secondary/embeddings.pkl`. Set `FORCE_REEXTRACT = True` above when the dataset or embedding configuration changes.

In [ ]:
def extract_embeddings(frame):
    embeddings, labels, paths, failures = [], [], [], []
    started = time.time()

    for number, row in enumerate(frame.itertuples(index=False), start=1):
        try:
            result = DeepFace.represent(
                img_path=row.image_path,
                model_name=MODEL_NAME,
                detector_backend=DETECTOR_BACKEND,
                enforce_detection=False,
            )
            if not result or "embedding" not in result[0]:
                raise ValueError("DeepFace returned no embedding")
            embeddings.append(result[0]["embedding"])
            labels.append(row.label)
            paths.append(row.image_path)
        except Exception as exc:
            failures.append({"image_path": row.image_path, "error": str(exc)})

        if number % 100 == 0 or number == len(frame):
            elapsed = time.time() - started
            print(f"Processed {number:4d}/{len(frame)} images ({elapsed:.1f}s)")

    if not embeddings:
        raise ValueError("No embeddings were extracted.")

    payload = {
        "embeddings": np.asarray(embeddings, dtype=np.float32),
        "labels": np.asarray(labels),
        "image_paths": paths,
        "model_name": MODEL_NAME,
        "detector_backend": DETECTOR_BACKEND,
    }
    return payload, pd.DataFrame(failures)


if EMBEDDINGS_PATH.exists() and not FORCE_REEXTRACT:
    embedding_data = joblib.load(EMBEDDINGS_PATH)
    failures_df = pd.DataFrame(columns=["image_path", "error"])
    print(f"Loaded cached embeddings from {EMBEDDINGS_PATH}")
else:
    embedding_data, failures_df = extract_embeddings(image_df)
    joblib.dump(embedding_data, EMBEDDINGS_PATH)
    failures_df.to_csv(REPORT_DIR / "embedding_failures.csv", index=False)
    print(f"Saved embeddings to {EMBEDDINGS_PATH}")

X = np.asarray(embedding_data["embeddings"], dtype=np.float32)
y = np.asarray(embedding_data["labels"]).astype(str)
embedding_paths = list(embedding_data["image_paths"])

if len(X) != len(y) or len(y) != len(embedding_paths):
    raise ValueError("Cached embedding arrays have inconsistent lengths.")
if len(np.unique(y)) < 2:
    raise ValueError("The extracted embeddings contain fewer than two classes.")

print(f"Embedding matrix: {X.shape}")
print(f"Classes: {len(np.unique(y))}")
print(f"Failures in this extraction run: {len(failures_df)}")

## 5. Encode labels, split the data, and train

L2 normalization makes the classifier compare embedding direction rather than raw magnitude. The fixed seed and stratification make the 80/20 split reproducible and preserve every class in both sets.

In [ ]:
X_normalized = normalize(X, norm="l2")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X_normalized,
    y_encoded,
    embedding_paths,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_encoded,
)

classifier = SVC(
    kernel="linear",
    probability=True,
    class_weight="balanced",
    random_state=SEED,
)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
y_probability = classifier.predict_proba(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")
print(f"Test accuracy:    {accuracy:.2%}")

## 6. Evaluate the classifier

In [ ]:
class_names = label_encoder.classes_
report_dict = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df)

matrix = confusion_matrix(y_test, y_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(18, 18))
ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
    ax=ax, cmap="Blues", colorbar=False, xticks_rotation=90, values_format="d"
)
ax.set_title("Secondary dataset confusion matrix")
plt.tight_layout()
plt.show()

true_labels = label_encoder.inverse_transform(y_test)
predicted_labels = label_encoder.inverse_transform(y_pred)
prediction_df = pd.DataFrame({
    "image_path": paths_test,
    "true_label": true_labels,
    "predicted_label": predicted_labels,
    "confidence": y_probability.max(axis=1),
})
prediction_df["correct"] = prediction_df["true_label"] == prediction_df["predicted_label"]
display(prediction_df.sort_values(["correct", "confidence"]).head(20))

## 7. Save the model and reports

These files live in secondary-specific directories and do not replace `models/face_classifier.pkl` or other production artifacts.

In [ ]:
joblib.dump(classifier, CLASSIFIER_PATH)
joblib.dump(label_encoder, ENCODER_PATH)

report_df.to_csv(REPORT_DIR / "classification_report.csv")
prediction_df.to_csv(REPORT_DIR / "test_predictions.csv", index=False)
np.savetxt(REPORT_DIR / "confusion_matrix.csv", matrix, delimiter=",", fmt="%d")

metadata = {
    "embedding_model": MODEL_NAME,
    "detector_backend": DETECTOR_BACKEND,
    "classifier": "SVC(kernel='linear', probability=True, class_weight='balanced')",
    "normalization": "L2",
    "random_seed": SEED,
    "test_size": TEST_SIZE,
    "total_embeddings": int(len(X)),
    "training_samples": int(len(X_train)),
    "testing_samples": int(len(X_test)),
    "classes": int(len(class_names)),
    "accuracy": float(accuracy),
}
with (REPORT_DIR / "training_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved:")
for path in [
    EMBEDDINGS_PATH,
    CLASSIFIER_PATH,
    ENCODER_PATH,
    REPORT_DIR / "classification_report.csv",
    REPORT_DIR / "test_predictions.csv",
    REPORT_DIR / "confusion_matrix.csv",
    REPORT_DIR / "training_metadata.json",
]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

## 8. Predict one image

The helper below applies the same FaceNet extraction and L2 normalization used during training.

In [ ]:
def predict_image(image_path):
    image_path = Path(image_path)
    result = DeepFace.represent(
        img_path=str(image_path),
        model_name=MODEL_NAME,
        detector_backend=DETECTOR_BACKEND,
        enforce_detection=False,
    )
    embedding = normalize(np.asarray([result[0]["embedding"]], dtype=np.float32), norm="l2")
    encoded_prediction = classifier.predict(embedding)[0]
    probabilities = classifier.predict_proba(embedding)[0]
    return {
        "predicted_label": label_encoder.inverse_transform([encoded_prediction])[0],
        "confidence": float(probabilities.max()),
    }

example_path = prediction_df.iloc[0]["image_path"]
print(f"Image: {example_path}")
print(predict_image(example_path))